In [ ]:
# =============================================================================
# Paper1_11_MultiMAE_Baseline_13x13
# CELL 1
#
# Environment + frozen-data audit
#
# Kernel required:
#     Python (paper1_ssl)
#
# Purpose:
#     Prepare a capacity-matched MultiMAE-style contemporary baseline
#     using the EXACT same frozen 13x13 spatial folds as PB-MMAE and
#     Vanilla MAE.
# =============================================================================

from pathlib import Path
import sys
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)


# -----------------------------------------------------------------------------
# 1. Environment
# -----------------------------------------------------------------------------

print("=" * 96)
print("MULTIMAE-STYLE BASELINE — ENVIRONMENT AUDIT")
print("=" * 96)

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("NumPy version     :", np.__version__)
print("PyTorch version   :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "CUDA device       :",
        torch.cuda.get_device_name(0)
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Selected device   :", DEVICE)


# -----------------------------------------------------------------------------
# 2. Project paths
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

PATCH_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "patches"
    / "13x13"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "03_outputs"
    / "modern_baselines"
    / "MultiMAE_13x13"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert PATCH_ROOT.exists(), PATCH_ROOT


print()
print("PROJECT_ROOT :", PROJECT_ROOT)
print("PATCH_ROOT   :", PATCH_ROOT)
print("OUTPUT_ROOT  :", OUTPUT_ROOT)


# -----------------------------------------------------------------------------
# 3. Frozen data dimensions
# -----------------------------------------------------------------------------

PATCH_SIZE = 13
N_SPATIAL_TOKENS = 13 * 13
N_CHANNELS = 14

# Same absolute spatial masking budget used by Vanilla MAE
N_MASKED_SPATIAL = 68

MASK_RATIO = (
    N_MASKED_SPATIAL
    /
    N_SPATIAL_TOKENS
)


print()
print("=" * 96)
print("FROZEN INPUT GEOMETRY")
print("=" * 96)

print("Patch size          :", PATCH_SIZE, "x", PATCH_SIZE)
print("Spatial tokens      :", N_SPATIAL_TOKENS)
print("Input channels      :", N_CHANNELS)
print("Masked spatial count:", N_MASKED_SPATIAL)
print(
    "Spatial mask ratio  :",
    f"{MASK_RATIO:.6f}"
)


assert N_SPATIAL_TOKENS == 169
assert N_MASKED_SPATIAL == 68


# -----------------------------------------------------------------------------
# 4. Same six modality families used throughout the study
#
# IMPORTANT:
# These are treated as generic modalities for this baseline.
# No equal 1/6 physics-balanced loss is applied.
# -----------------------------------------------------------------------------

MODALITY_CHANNELS = {

    "MAGNETIC": [
        0
    ],

    "GRAVITY": [
        1,
        2,
        3
    ],

    "RADIOMETRIC": [
        4,
        5,
        6
    ],

    "THERMAL": [
        7
    ],

    "STRUCTURAL": [
        8,
        9,
        10,
        11
    ],

    "TERRAIN": [
        12,
        13
    ],
}


print()
print("=" * 96)
print("MULTIMODAL INPUT ORGANIZATION")
print("=" * 96)

for name, channels in (
    MODALITY_CHANNELS.items()
):

    print(
        f"{name:<14}: "
        f"{len(channels)} channel(s) "
        f"-> {channels}"
    )


all_channels = sorted(
    [
        channel
        for channels
        in MODALITY_CHANNELS.values()
        for channel
        in channels
    ]
)

assert all_channels == list(
    range(N_CHANNELS)
)


# -----------------------------------------------------------------------------
# 5. Audit frozen Fold-3 development arrays
# -----------------------------------------------------------------------------

FOLD3_DIR = (
    PATCH_ROOT
    / "Fold_3"
)

TRAIN_FILE = (
    FOLD3_DIR
    / "Fold3_TRAIN_13x13_14ch.npy"
)

VAL_FILE = (
    FOLD3_DIR
    / "Fold3_VALIDATION_13x13_14ch.npy"
)

TRAIN_META_FILE = (
    FOLD3_DIR
    / "Fold3_TRAIN_13x13_metadata.csv"
)

VAL_META_FILE = (
    FOLD3_DIR
    / "Fold3_VALIDATION_13x13_metadata.csv"
)


for path in [
    TRAIN_FILE,
    VAL_FILE,
    TRAIN_META_FILE,
    VAL_META_FILE,
]:

    assert path.exists(), path


X_train = np.load(
    TRAIN_FILE,
    mmap_mode="r"
)

X_val = np.load(
    VAL_FILE,
    mmap_mode="r"
)

train_meta = pd.read_csv(
    TRAIN_META_FILE
)

val_meta = pd.read_csv(
    VAL_META_FILE
)


print()
print("=" * 96)
print("FOLD 3 — DEVELOPMENT DATA AUDIT")
print("=" * 96)

print("TRAIN shape :", X_train.shape)
print("VAL shape   :", X_val.shape)

print(
    "TRAIN finite:",
    np.isfinite(
        np.asarray(X_train)
    ).all()
)

print(
    "VAL finite  :",
    np.isfinite(
        np.asarray(X_val)
    ).all()
)

print("TRAIN metadata:", len(train_meta))
print("VAL metadata  :", len(val_meta))


assert X_train.shape == (
    455,
    13,
    13,
    14
)

assert X_val.shape == (
    392,
    13,
    13,
    14
)

assert len(train_meta) == 455
assert len(val_meta) == 392


# -----------------------------------------------------------------------------
# 6. Audit all frozen spatial folds
# -----------------------------------------------------------------------------

expected_counts = {

    1: (
        990,
        199
    ),

    2: (
        626,
        316
    ),

    3: (
        455,
        392
    ),

    4: (
        571,
        364
    ),

    5: (
        935,
        206
    ),
}


fold_records = []


for fold, (
    expected_train,
    expected_val
) in expected_counts.items():

    fold_dir = (
        PATCH_ROOT
        / f"Fold_{fold}"
    )

    train_path = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"TRAIN_13x13_14ch.npy"
        )
    )

    val_path = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"VALIDATION_13x13_14ch.npy"
        )
    )


    assert train_path.exists(), train_path
    assert val_path.exists(), val_path


    train_arr = np.load(
        train_path,
        mmap_mode="r"
    )

    val_arr = np.load(
        val_path,
        mmap_mode="r"
    )


    assert train_arr.shape == (
        expected_train,
        13,
        13,
        14
    )

    assert val_arr.shape == (
        expected_val,
        13,
        13,
        14
    )


    fold_records.append(
        {
            "Fold":
                fold,

            "Train_n":
                train_arr.shape[0],

            "Validation_n":
                val_arr.shape[0],

            "Channels":
                train_arr.shape[-1],

            "Patch":
                f"{train_arr.shape[1]}x"
                f"{train_arr.shape[2]}"
        }
    )


fold_audit_df = pd.DataFrame(
    fold_records
)


print()
print("=" * 96)
print("FROZEN SPATIAL-FOLD AUDIT")
print("=" * 96)

display(
    fold_audit_df
)


# -----------------------------------------------------------------------------
# 7. Baseline design locks
# -----------------------------------------------------------------------------

MULTIMAE_LATENT_DIM = 64

TARGET_PARAMETER_RANGE = (
    195_000,
    220_000
)


print()
print("=" * 96)
print("MULTIMAE-STYLE DESIGN LOCKS")
print("=" * 96)

print(
    "Output latent dimension :",
    MULTIMAE_LATENT_DIM
)

print(
    "Target parameter range  :",
    TARGET_PARAMETER_RANGE
)

print(
    "Reference Vanilla MAE   :",
    "207,182 parameters"
)

print(
    "Reference PB-MMAE       :",
    "206,190 parameters"
)


print()
print("=" * 96)
print(
    "STATUS: MULTIMAE-STYLE INPUT AUDIT PASSED"
)
print("=" * 96)

In [ ]:
# =============================================================================
# Paper1_11_MultiMAE_Baseline_13x13
# CELL 2
#
# MultiMAE-style architecture + parameter-count + smoke-test audit
#
# DESIGN:
#   - Six modality-specific input adapters
#   - Independent masking within each modality
#   - Exactly 68 / 169 masked spatial positions PER modality
#   - Shared 64-D Transformer encoder
#   - Shared Transformer decoder
#   - Modality-specific output heads
#   - Generic masked-element MSE
#
# EXCLUDED BY DESIGN:
#   - NO PB-MMAE equal 1/6 family weighting
#   - NO whole-modality dropout at p=0.30
#   - NO family-specific encoders
#
# This is a MultiMAE-style geophysical baseline, not an exact reproduction
# of the original MultiMAE implementation.
# =============================================================================

import math

import numpy as np
import torch
import torch.nn as nn


# -----------------------------------------------------------------------------
# 1. Frozen architecture configuration
# -----------------------------------------------------------------------------

MM_EMBED_DIM = 64

MM_ENCODER_DEPTH = 4
MM_ENCODER_HEADS = 4
MM_ENCODER_MLP_DIM = 128

MM_DECODER_DIM = 64
MM_DECODER_DEPTH = 2
MM_DECODER_HEADS = 4
MM_DECODER_MLP_DIM = 128

MM_DROPOUT = 0.0

MM_N_MODALITIES = len(
    MODALITY_CHANNELS
)

MM_N_MASKED_PER_MODALITY = (
    N_MASKED_SPATIAL
)


print("=" * 96)
print("MULTIMAE-STYLE ARCHITECTURE CONFIGURATION")
print("=" * 96)

print("Embedding dimension       :", MM_EMBED_DIM)
print("Encoder depth             :", MM_ENCODER_DEPTH)
print("Encoder heads             :", MM_ENCODER_HEADS)
print("Encoder FF dimension      :", MM_ENCODER_MLP_DIM)

print("Decoder depth             :", MM_DECODER_DEPTH)
print("Decoder heads             :", MM_DECODER_HEADS)
print("Decoder FF dimension      :", MM_DECODER_MLP_DIM)

print("Modalities                :", MM_N_MODALITIES)
print(
    "Masked positions/modality:",
    MM_N_MASKED_PER_MODALITY
)

print(
    "Mask fraction/modality    :",
    f"{MM_N_MASKED_PER_MODALITY / N_SPATIAL_TOKENS:.6f}"
)


# -----------------------------------------------------------------------------
# 2. Fixed 2-D sine/cosine positional encoding
# -----------------------------------------------------------------------------

def _sincos_1d(
    positions,
    dim
):
    """
    positions : 1-D tensor
    dim       : even embedding dimension
    """

    assert dim % 2 == 0

    half = dim // 2

    omega = torch.arange(
        half,
        dtype=torch.float32
    )

    omega = (
        1.0
        /
        (
            10000.0
            **
            (
                omega
                /
                half
            )
        )
    )

    out = (
        positions[:, None]
        *
        omega[None, :]
    )

    return torch.cat(
        [
            torch.sin(out),
            torch.cos(out)
        ],
        dim=1
    )


def build_2d_sincos_position_encoding(
    height,
    width,
    dim
):
    """
    Returns:
        [1, height*width, dim]
    """

    assert dim % 4 == 0

    yy, xx = torch.meshgrid(
        torch.arange(
            height,
            dtype=torch.float32
        ),
        torch.arange(
            width,
            dtype=torch.float32
        ),
        indexing="ij"
    )

    yy = yy.reshape(-1)
    xx = xx.reshape(-1)

    enc_y = _sincos_1d(
        yy,
        dim // 2
    )

    enc_x = _sincos_1d(
        xx,
        dim // 2
    )

    pos = torch.cat(
        [
            enc_y,
            enc_x
        ],
        dim=1
    )

    assert pos.shape == (
        height * width,
        dim
    )

    return pos.unsqueeze(0)


# -----------------------------------------------------------------------------
# 3. MultiMAE-style model
# -----------------------------------------------------------------------------

class MultiMAEStyle13x13(
    nn.Module
):

    def __init__(
        self,
        patch_size=13,
        modality_channels=None,
        embed_dim=64,
        encoder_depth=4,
        encoder_heads=4,
        encoder_mlp_dim=128,
        decoder_dim=64,
        decoder_depth=2,
        decoder_heads=4,
        decoder_mlp_dim=128,
        n_masked_per_modality=68,
        dropout=0.0,
    ):

        super().__init__()


        if modality_channels is None:

            modality_channels = (
                MODALITY_CHANNELS
            )


        self.patch_size = patch_size

        self.n_spatial_tokens = (
            patch_size
            *
            patch_size
        )

        self.embed_dim = embed_dim

        self.decoder_dim = decoder_dim

        self.n_masked_per_modality = (
            n_masked_per_modality
        )


        self.modality_names = list(
            modality_channels.keys()
        )

        self.modality_channels = {
            name:
                list(channels)

            for name, channels
            in modality_channels.items()
        }

        self.n_modalities = len(
            self.modality_names
        )


        assert (
            self.n_masked_per_modality
            <
            self.n_spatial_tokens
        )

        assert (
            embed_dim
            ==
            decoder_dim
        ), (
            "This capacity-matched baseline "
            "uses encoder_dim = decoder_dim = 64."
        )


        # ---------------------------------------------------------------------
        # Modality-specific input adapters
        #
        # Each modality maps its own physical-variable vector at each spatial
        # location into the common 64-D token space.
        # ---------------------------------------------------------------------

        self.input_adapters = (
            nn.ModuleDict()
        )


        for name in self.modality_names:

            n_channels = len(
                self.modality_channels[
                    name
                ]
            )

            self.input_adapters[
                name
            ] = nn.Linear(
                n_channels,
                embed_dim
            )


        # ---------------------------------------------------------------------
        # Learned modality identity embeddings
        # ---------------------------------------------------------------------

        self.modality_embeddings = (
            nn.Parameter(
                torch.zeros(
                    self.n_modalities,
                    embed_dim
                )
            )
        )


        # ---------------------------------------------------------------------
        # Fixed spatial position encoding
        # ---------------------------------------------------------------------

        position_encoding = (
            build_2d_sincos_position_encoding(
                patch_size,
                patch_size,
                embed_dim
            )
        )


        self.register_buffer(
            "position_encoding",
            position_encoding,
            persistent=True
        )


        # ---------------------------------------------------------------------
        # Shared Transformer encoder
        # ---------------------------------------------------------------------

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=encoder_heads,
                dim_feedforward=encoder_mlp_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )


        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=encoder_depth
            )
        )


        # ---------------------------------------------------------------------
        # Decoder projection
        # ---------------------------------------------------------------------

        self.decoder_projection = (
            nn.Linear(
                embed_dim,
                decoder_dim
            )
        )


        # ---------------------------------------------------------------------
        # One learned mask indicator per modality.
        #
        # These are supplied to the decoder only. They identify which
        # modalities are missing at a spatial location without exposing their
        # target values to the encoder.
        # ---------------------------------------------------------------------

        self.modality_mask_tokens = (
            nn.Parameter(
                torch.zeros(
                    self.n_modalities,
                    decoder_dim
                )
            )
        )


        # ---------------------------------------------------------------------
        # Shared Transformer decoder
        # ---------------------------------------------------------------------

        decoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=decoder_dim,
                nhead=decoder_heads,
                dim_feedforward=decoder_mlp_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )


        self.decoder = (
            nn.TransformerEncoder(
                decoder_layer,
                num_layers=decoder_depth
            )
        )


        # ---------------------------------------------------------------------
        # Modality-specific reconstruction heads
        # ---------------------------------------------------------------------

        self.output_heads = (
            nn.ModuleDict()
        )


        for name in self.modality_names:

            n_channels = len(
                self.modality_channels[
                    name
                ]
            )

            self.output_heads[
                name
            ] = nn.Linear(
                decoder_dim,
                n_channels
            )


        # ---------------------------------------------------------------------
        # Initialization of learned modality identifiers
        # ---------------------------------------------------------------------

        nn.init.trunc_normal_(
            self.modality_embeddings,
            std=0.02
        )

        nn.init.trunc_normal_(
            self.modality_mask_tokens,
            std=0.02
        )


    # =========================================================================
    # Independent modality-specific masks
    # =========================================================================

    def generate_masks(
        self,
        batch_size,
        device
    ):
        """
        Returns boolean tensor:

            [B, M, 169]

        Exactly 68 positions are masked independently for every modality
        and every sample.
        """

        random_scores = torch.rand(
            batch_size,
            self.n_modalities,
            self.n_spatial_tokens,
            device=device
        )


        order = torch.argsort(
            random_scores,
            dim=-1
        )


        masks = torch.zeros(
            batch_size,
            self.n_modalities,
            self.n_spatial_tokens,
            dtype=torch.bool,
            device=device
        )


        selected = order[
            :,
            :,
            :self.n_masked_per_modality
        ]


        masks.scatter_(
            dim=2,
            index=selected,
            value=True
        )


        return masks


    # =========================================================================
    # Build encoder tokens from VISIBLE modality information only
    # =========================================================================

    def build_encoder_tokens(
        self,
        x_flat,
        masks
    ):
        """
        x_flat:
            [B, 169, 14]

        masks:
            [B, 6, 169]

        The six modality-specific projections are fused at each spatial
        position using only visible modalities.

        Division by the number of visible modalities keeps token scale
        approximately stable as masking patterns change.
        """

        B = x_flat.shape[0]

        token_sum = torch.zeros(
            B,
            self.n_spatial_tokens,
            self.embed_dim,
            dtype=x_flat.dtype,
            device=x_flat.device
        )


        visible_count = torch.zeros(
            B,
            self.n_spatial_tokens,
            1,
            dtype=x_flat.dtype,
            device=x_flat.device
        )


        for m_idx, name in enumerate(
            self.modality_names
        ):

            channels = (
                self.modality_channels[
                    name
                ]
            )


            x_modality = x_flat[
                :,
                :,
                channels
            ]


            z = (
                self.input_adapters[
                    name
                ](
                    x_modality
                )
            )


            z = (
                z
                +
                self.modality_embeddings[
                    m_idx
                ][
                    None,
                    None,
                    :
                ]
            )


            visible = (
                ~masks[
                    :,
                    m_idx,
                    :
                ]
            ).unsqueeze(
                -1
            )


            visible_float = (
                visible.to(
                    dtype=x_flat.dtype
                )
            )


            token_sum = (
                token_sum
                +
                z
                *
                visible_float
            )


            visible_count = (
                visible_count
                +
                visible_float
            )


        # If all six modalities happen to be masked at one position,
        # the data contribution remains zero and only position information
        # enters the encoder.
        tokens = (
            token_sum
            /
            visible_count.clamp_min(
                1.0
            )
        )


        tokens = (
            tokens
            +
            self.position_encoding.to(
                dtype=tokens.dtype,
                device=tokens.device
            )
        )


        return tokens


    # =========================================================================
    # Forward masked-reconstruction pass
    # =========================================================================

    def forward(
        self,
        x,
        masks=None
    ):

        assert x.ndim == 4

        assert x.shape[1:] == (
            self.patch_size,
            self.patch_size,
            14
        )


        B = x.shape[0]


        x_flat = x.reshape(
            B,
            self.n_spatial_tokens,
            14
        )


        if masks is None:

            masks = self.generate_masks(
                B,
                x.device
            )


        assert masks.shape == (
            B,
            self.n_modalities,
            self.n_spatial_tokens
        )


        # ---------------------------------------------------------------------
        # Encoder
        # ---------------------------------------------------------------------

        encoder_tokens = (
            self.build_encoder_tokens(
                x_flat,
                masks
            )
        )


        encoded = self.encoder(
            encoder_tokens
        )


        # ---------------------------------------------------------------------
        # Decoder input
        # ---------------------------------------------------------------------

        decoded_input = (
            self.decoder_projection(
                encoded
            )
        )


        # Tell the decoder WHICH modalities are masked at each position.
        mask_signal = torch.zeros_like(
            decoded_input
        )


        masked_count = torch.zeros(
            B,
            self.n_spatial_tokens,
            1,
            dtype=x.dtype,
            device=x.device
        )


        for m_idx in range(
            self.n_modalities
        ):

            masked = (
                masks[
                    :,
                    m_idx,
                    :
                ]
                .unsqueeze(-1)
                .to(
                    dtype=x.dtype
                )
            )


            mask_signal = (
                mask_signal
                +
                masked
                *
                self.modality_mask_tokens[
                    m_idx
                ][
                    None,
                    None,
                    :
                ]
            )


            masked_count = (
                masked_count
                +
                masked
            )


        mask_signal = (
            mask_signal
            /
            masked_count.clamp_min(
                1.0
            )
        )


        decoded_input = (
            decoded_input
            +
            mask_signal
            +
            self.position_encoding.to(
                dtype=decoded_input.dtype,
                device=decoded_input.device
            )
        )


        decoded = self.decoder(
            decoded_input
        )


        # ---------------------------------------------------------------------
        # Modality-specific reconstruction + generic scalar-element MSE
        #
        # NO equal-family weighting.
        #
        # Every masked scalar value contributes equally to the objective.
        # Consequently, modalities with more channels contribute more scalar
        # reconstruction terms.
        # ---------------------------------------------------------------------

        prediction = torch.zeros(
            B,
            self.n_spatial_tokens,
            14,
            dtype=x.dtype,
            device=x.device
        )


        squared_error_sum = torch.zeros(
            (),
            dtype=x.dtype,
            device=x.device
        )


        masked_scalar_count = torch.zeros(
            (),
            dtype=x.dtype,
            device=x.device
        )


        for m_idx, name in enumerate(
            self.modality_names
        ):

            channels = (
                self.modality_channels[
                    name
                ]
            )


            pred_modality = (
                self.output_heads[
                    name
                ](
                    decoded
                )
            )


            target_modality = x_flat[
                :,
                :,
                channels
            ]


            prediction[
                :,
                :,
                channels
            ] = pred_modality


            masked = (
                masks[
                    :,
                    m_idx,
                    :
                ]
                .unsqueeze(
                    -1
                )
            )


            mask_float = masked.to(
                dtype=x.dtype
            )


            squared_error = (
                pred_modality
                -
                target_modality
            ) ** 2


            squared_error_sum = (
                squared_error_sum
                +
                (
                    squared_error
                    *
                    mask_float
                ).sum()
            )


            masked_scalar_count = (
                masked_scalar_count
                +
                mask_float.sum()
                *
                len(
                    channels
                )
            )


        loss = (
            squared_error_sum
            /
            masked_scalar_count
        )


        prediction = prediction.reshape(
            B,
            self.patch_size,
            self.patch_size,
            14
        )


        return {

            "loss":
                loss,

            "prediction":
                prediction,

            "masks":
                masks,

            "encoded_tokens":
                encoded
        }


    # =========================================================================
    # Full-input representation for downstream clustering
    # =========================================================================

    @torch.no_grad()
    def encode_full(
        self,
        x
    ):
        """
        No masking.

        All six modalities contribute to every spatial token.

        Final representation:
            mean pool over 169 encoded spatial tokens -> 64-D.
        """

        assert x.ndim == 4

        B = x.shape[0]


        x_flat = x.reshape(
            B,
            self.n_spatial_tokens,
            14
        )


        masks = torch.zeros(
            B,
            self.n_modalities,
            self.n_spatial_tokens,
            dtype=torch.bool,
            device=x.device
        )


        tokens = self.build_encoder_tokens(
            x_flat,
            masks
        )


        encoded = self.encoder(
            tokens
        )


        latent = encoded.mean(
            dim=1
        )


        return latent


# -----------------------------------------------------------------------------
# 4. Instantiate model
# -----------------------------------------------------------------------------

torch.manual_seed(
    2026
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        2026
    )


multimae_model = (
    MultiMAEStyle13x13(
        patch_size=PATCH_SIZE,
        modality_channels=MODALITY_CHANNELS,
        embed_dim=MM_EMBED_DIM,
        encoder_depth=MM_ENCODER_DEPTH,
        encoder_heads=MM_ENCODER_HEADS,
        encoder_mlp_dim=MM_ENCODER_MLP_DIM,
        decoder_dim=MM_DECODER_DIM,
        decoder_depth=MM_DECODER_DEPTH,
        decoder_heads=MM_DECODER_HEADS,
        decoder_mlp_dim=MM_DECODER_MLP_DIM,
        n_masked_per_modality=(
            MM_N_MASKED_PER_MODALITY
        ),
        dropout=MM_DROPOUT
    )
    .to(
        DEVICE
    )
)


# -----------------------------------------------------------------------------
# 5. Parameter-count audit
# -----------------------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in multimae_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in multimae_model.parameters()
    if p.requires_grad
)


PBMMAE_PARAMETERS = 206_190
VANILLA_MAE_PARAMETERS = 207_182


print()
print("=" * 96)
print("PARAMETER-COUNT AUDIT")
print("=" * 96)

print(
    "MultiMAE-style total     :",
    f"{total_parameters:,}"
)

print(
    "MultiMAE-style trainable :",
    f"{trainable_parameters:,}"
)

print(
    "PB-MMAE                  :",
    f"{PBMMAE_PARAMETERS:,}"
)

print(
    "Vanilla MAE              :",
    f"{VANILLA_MAE_PARAMETERS:,}"
)


print(
    "Difference vs PB-MMAE    :",
    f"{trainable_parameters - PBMMAE_PARAMETERS:+,}",
    f"({100 * (trainable_parameters - PBMMAE_PARAMETERS) / PBMMAE_PARAMETERS:+.3f}%)"
)

print(
    "Difference vs Vanilla MAE:",
    f"{trainable_parameters - VANILLA_MAE_PARAMETERS:+,}",
    f"({100 * (trainable_parameters - VANILLA_MAE_PARAMETERS) / VANILLA_MAE_PARAMETERS:+.3f}%)"
)


assert (
    TARGET_PARAMETER_RANGE[0]
    <=
    trainable_parameters
    <=
    TARGET_PARAMETER_RANGE[1]
)


# -----------------------------------------------------------------------------
# 6. Smoke test using actual Fold-3 samples
# -----------------------------------------------------------------------------

x_smoke = torch.from_numpy(
    np.array(
        X_train[:2],
        dtype=np.float32,
        copy=True
    )
).to(
    DEVICE
)


multimae_model.train()


smoke_output = (
    multimae_model(
        x_smoke
    )
)


smoke_loss = (
    smoke_output[
        "loss"
    ]
)

smoke_prediction = (
    smoke_output[
        "prediction"
    ]
)

smoke_masks = (
    smoke_output[
        "masks"
    ]
)


print()
print("=" * 96)
print("MASKED-RECONSTRUCTION SMOKE TEST")
print("=" * 96)

print(
    "Input shape      :",
    tuple(
        x_smoke.shape
    )
)

print(
    "Prediction shape :",
    tuple(
        smoke_prediction.shape
    )
)

print(
    "Mask shape       :",
    tuple(
        smoke_masks.shape
    )
)

print(
    "Loss             :",
    float(
        smoke_loss.item()
    )
)


# Each sample × modality must contain exactly 68 masked positions.
masked_counts = (
    smoke_masks
    .sum(
        dim=2
    )
    .detach()
    .cpu()
    .numpy()
)


print()
print(
    "Masked counts per sample/modality:"
)

print(
    masked_counts
)


assert smoke_prediction.shape == (
    2,
    13,
    13,
    14
)

assert smoke_masks.shape == (
    2,
    6,
    169
)

assert np.all(
    masked_counts
    ==
    MM_N_MASKED_PER_MODALITY
)

assert torch.isfinite(
    smoke_loss
)


# -----------------------------------------------------------------------------
# 7. Full-input latent smoke test
# -----------------------------------------------------------------------------

multimae_model.eval()


with torch.no_grad():

    z_full = (
        multimae_model.encode_full(
            x_smoke
        )
    )


print()
print("=" * 96)
print("FULL-INPUT LATENT SMOKE TEST")
print("=" * 96)

print(
    "Latent shape :",
    tuple(
        z_full.shape
    )
)

print(
    "Latent finite:",
    bool(
        torch.isfinite(
            z_full
        ).all()
    )
)


assert z_full.shape == (
    2,
    64
)

assert torch.isfinite(
    z_full
).all()


# -----------------------------------------------------------------------------
# 8. Save architecture audit
# -----------------------------------------------------------------------------

architecture_audit = pd.DataFrame(
    [
        {
            "Model":
                "MultiMAE-style",

            "Trainable_parameters":
                trainable_parameters,

            "Patch_size":
                "13x13",

            "Input_channels":
                14,

            "Modalities":
                6,

            "Masked_positions_per_modality":
                MM_N_MASKED_PER_MODALITY,

            "Mask_fraction_per_modality":
                (
                    MM_N_MASKED_PER_MODALITY
                    /
                    N_SPATIAL_TOKENS
                ),

            "Latent_dimension":
                MM_EMBED_DIM,

            "Encoder_depth":
                MM_ENCODER_DEPTH,

            "Decoder_depth":
                MM_DECODER_DEPTH,

            "Equal_family_loss":
                False,

            "Whole_modality_dropout":
                False,

            "Masking":
                (
                    "Independent spatial masking "
                    "within each modality"
                ),
        }
    ]
)


ARCHITECTURE_AUDIT_FILE = (
    OUTPUT_ROOT
    / "MultiMAE_13x13_architecture_audit.csv"
)


architecture_audit.to_csv(
    ARCHITECTURE_AUDIT_FILE,
    index=False
)


print()
print("=" * 96)
print(
    "STATUS: MULTIMAE-STYLE ARCHITECTURE AUDIT PASSED"
)
print("=" * 96)

print(
    "Architecture audit saved:"
)

print(
    ARCHITECTURE_AUDIT_FILE
)

In [ ]:
# =============================================================================
# Paper1_11_MultiMAE_Baseline_13x13
# CELL 3
#
# Fold-3 DEVELOPMENT training sweep for MultiMAE-style baseline
#
# TUNED HERE ONLY:
#   - learning rate
#   - retained epoch
#
# FIXED:
#   - architecture
#   - weight decay = 1e-4
#   - batch size = 32
#   - max epochs = 300
#   - masking = 68 / 169 independently per modality
#
# Selection criterion:
#   minimum held-out Fold-3 masked reconstruction MSE
#
# NO clustering or external geological information is used.
# =============================================================================

from pathlib import Path
import copy
import json
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)


# -----------------------------------------------------------------------------
# 1. Development configuration
# -----------------------------------------------------------------------------

LR_CANDIDATES = [
    1e-4,
    3e-4,
    1e-3
]

MAX_EPOCHS = 300

WEIGHT_DECAY = 1e-4

BATCH_SIZE = 32

TRAIN_SEED = 2026

VAL_MASK_SEED = 123456


TUNING_DIR = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "MultiMAE_training_tuning"
)

TUNING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 96)
print("MULTIMAE-STYLE — FOLD-3 DEVELOPMENT TRAINING SWEEP")
print("=" * 96)

print("LR candidates :", LR_CANDIDATES)
print("Max epochs    :", MAX_EPOCHS)
print("Weight decay  :", WEIGHT_DECAY)
print("Batch size    :", BATCH_SIZE)
print("Train seed    :", TRAIN_SEED)
print("Val-mask seed :", VAL_MASK_SEED)
print("Device        :", DEVICE)


# -----------------------------------------------------------------------------
# 2. Dataset
# -----------------------------------------------------------------------------

class MultiMAEPatchDataset(
    Dataset
):

    def __init__(
        self,
        npy_path
    ):

        self.x = np.load(
            npy_path,
            mmap_mode="r"
        )


    def __len__(self):

        return self.x.shape[0]


    def __getitem__(
        self,
        idx
    ):

        sample = np.array(
            self.x[idx],
            dtype=np.float32,
            copy=True
        )

        return torch.from_numpy(
            sample
        )


train_ds = MultiMAEPatchDataset(
    TRAIN_FILE
)

val_ds = MultiMAEPatchDataset(
    VAL_FILE
)


assert len(train_ds) == 455
assert len(val_ds) == 392


# -----------------------------------------------------------------------------
# 3. Reproducibility helper
# -----------------------------------------------------------------------------

def reset_multimae_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# 4. Fresh-model constructor
# -----------------------------------------------------------------------------

def build_fresh_multimae():

    model = MultiMAEStyle13x13(
        patch_size=PATCH_SIZE,
        modality_channels=MODALITY_CHANNELS,
        embed_dim=MM_EMBED_DIM,
        encoder_depth=MM_ENCODER_DEPTH,
        encoder_heads=MM_ENCODER_HEADS,
        encoder_mlp_dim=MM_ENCODER_MLP_DIM,
        decoder_dim=MM_DECODER_DIM,
        decoder_depth=MM_DECODER_DEPTH,
        decoder_heads=MM_DECODER_HEADS,
        decoder_mlp_dim=MM_DECODER_MLP_DIM,
        n_masked_per_modality=(
            MM_N_MASKED_PER_MODALITY
        ),
        dropout=MM_DROPOUT
    )

    return model.to(
        DEVICE
    )


# -----------------------------------------------------------------------------
# 5. Fixed validation-mask loss
#
# We reset and then restore RNG states so every epoch and every LR candidate
# is evaluated using the same deterministic validation masking sequence.
# -----------------------------------------------------------------------------

@torch.no_grad()
def fixed_multimae_validation_loss(
    model,
    loader,
    seed=VAL_MASK_SEED
):

    model.eval()


    cpu_state = (
        torch.get_rng_state()
    )


    cuda_state = None

    if torch.cuda.is_available():

        cuda_state = (
            torch.cuda.get_rng_state_all()
        )


    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    total_loss = 0.0

    total_n = 0


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )


        output = model(
            batch
        )


        loss = output[
            "loss"
        ]


        n = batch.shape[0]


        total_loss += (
            float(
                loss.item()
            )
            *
            n
        )


        total_n += n


    # Restore RNG state so validation does not alter training randomness.
    torch.set_rng_state(
        cpu_state
    )


    if (
        torch.cuda.is_available()
        and cuda_state is not None
    ):

        torch.cuda.set_rng_state_all(
            cuda_state
        )


    return (
        total_loss
        /
        total_n
    )


# -----------------------------------------------------------------------------
# 6. Development loaders
# -----------------------------------------------------------------------------

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)


# -----------------------------------------------------------------------------
# 7. Learning-rate sweep
# -----------------------------------------------------------------------------

all_histories = []

selection_records = []


for lr in LR_CANDIDATES:

    print()
    print("=" * 96)
    print(
        f"DEVELOPMENT RUN — LR = {lr:g}"
    )
    print("=" * 96)


    # -------------------------------------------------------------------------
    # Fresh deterministic model and shuffled TRAIN loader
    # -------------------------------------------------------------------------

    reset_multimae_seed(
        TRAIN_SEED
    )


    train_generator = (
        torch.Generator()
    )

    train_generator.manual_seed(
        TRAIN_SEED
    )


    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        generator=train_generator
    )


    model = build_fresh_multimae()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )


    best_val_loss = np.inf

    best_epoch = None

    best_train_loss = None

    best_state_dict = None


    run_records = []


    run_start = time.time()


    # -------------------------------------------------------------------------
    # Train full 300 epochs
    #
    # Fold 3 is the designated development fold, so epoch selection is allowed
    # here. Outer-fold epochs will later be frozen to the selected value.
    # -------------------------------------------------------------------------

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()


        total_train_loss = 0.0

        total_train_n = 0


        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            output = model(
                batch
            )


            loss = output[
                "loss"
            ]


            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    f"Non-finite loss at "
                    f"LR={lr}, epoch={epoch}"
                )


            loss.backward()

            optimizer.step()


            n = batch.shape[0]


            total_train_loss += (
                float(
                    loss.item()
                )
                *
                n
            )


            total_train_n += n


        train_loss = (
            total_train_loss
            /
            total_train_n
        )


        val_loss = (
            fixed_multimae_validation_loss(
                model,
                val_loader,
                seed=VAL_MASK_SEED
            )
        )


        run_records.append(
            {
                "Learning_rate":
                    lr,

                "Epoch":
                    epoch,

                "Train_masked_MSE":
                    train_loss,

                "Validation_masked_MSE":
                    val_loss
            }
        )


        # ---------------------------------------------------------------------
        # Development checkpoint selection
        # ---------------------------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = (
                val_loss
            )

            best_epoch = (
                epoch
            )

            best_train_loss = (
                train_loss
            )

            best_state_dict = {
                key:
                    value.detach()
                    .cpu()
                    .clone()

                for key, value
                in model.state_dict().items()
            }


        # ---------------------------------------------------------------------
        # Compact progress reporting
        # ---------------------------------------------------------------------

        if (
            epoch == 1
            or
            epoch % 25 == 0
            or
            epoch == MAX_EPOCHS
        ):

            print(
                f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
                f"train={train_loss:.8f} | "
                f"val={val_loss:.8f} | "
                f"best={best_val_loss:.8f} "
                f"@ {best_epoch}"
            )


    elapsed = (
        time.time()
        -
        run_start
    )


    # -------------------------------------------------------------------------
    # Save complete LR history
    # -------------------------------------------------------------------------

    history_df = pd.DataFrame(
        run_records
    )


    lr_tag = (
        f"{lr:.0e}"
        .replace(
            "-",
            "m"
        )
    )


    history_file = (
        TUNING_DIR
        / (
            f"MultiMAE_Fold3_"
            f"LR_{lr_tag}_history.csv"
        )
    )


    history_df.to_csv(
        history_file,
        index=False
    )


    # -------------------------------------------------------------------------
    # Save BEST Fold-3 checkpoint for this LR
    # -------------------------------------------------------------------------

    checkpoint_file = (
        TUNING_DIR
        / (
            f"MultiMAE_Fold3_"
            f"LR_{lr_tag}_BEST.pt"
        )
    )


    torch.save(
        {
            "model_state_dict":
                best_state_dict,

            "learning_rate":
                lr,

            "best_epoch":
                best_epoch,

            "best_train_masked_MSE":
                best_train_loss,

            "best_validation_masked_MSE":
                best_val_loss,

            "weight_decay":
                WEIGHT_DECAY,

            "batch_size":
                BATCH_SIZE,

            "train_seed":
                TRAIN_SEED,

            "validation_mask_seed":
                VAL_MASK_SEED,

            "selection_rule":
                (
                    "Minimum Fold-3 held-out "
                    "masked reconstruction MSE"
                )
        },
        checkpoint_file
    )


    selection_records.append(
        {
            "Learning_rate":
                lr,

            "Best_epoch":
                best_epoch,

            "Train_MSE_at_best_epoch":
                best_train_loss,

            "Validation_MSE_at_best_epoch":
                best_val_loss,

            "Training_seconds":
                elapsed,

            "Checkpoint":
                str(
                    checkpoint_file
                )
        }
    )


    all_histories.append(
        history_df
    )


    print()
    print(
        f"LR={lr:g} COMPLETE"
    )

    print(
        f"Best epoch     : "
        f"{best_epoch}"
    )

    print(
        f"Train MSE      : "
        f"{best_train_loss:.8f}"
    )

    print(
        f"Validation MSE : "
        f"{best_val_loss:.8f}"
    )

    print(
        f"Elapsed        : "
        f"{elapsed:.2f} s"
    )


# -----------------------------------------------------------------------------
# 8. Select frozen development configuration
# -----------------------------------------------------------------------------

selection_df = pd.DataFrame(
    selection_records
)


selection_df = (
    selection_df
    .sort_values(
        [
            "Validation_MSE_at_best_epoch",
            "Learning_rate"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


winner = (
    selection_df.iloc[0]
)


FROZEN_MULTIMAE_LR = float(
    winner[
        "Learning_rate"
    ]
)

FROZEN_MULTIMAE_EPOCHS = int(
    winner[
        "Best_epoch"
    ]
)

FROZEN_MULTIMAE_WEIGHT_DECAY = (
    WEIGHT_DECAY
)

FROZEN_MULTIMAE_BATCH_SIZE = (
    BATCH_SIZE
)


print()
print("=" * 96)
print("MULTIMAE-STYLE — FOLD-3 DEVELOPMENT SELECTION")
print("=" * 96)

display(
    selection_df.round(
        {
            "Learning_rate":
                6,

            "Train_MSE_at_best_epoch":
                8,

            "Validation_MSE_at_best_epoch":
                8,

            "Training_seconds":
                2
        }
    )
)


print()
print(
    "FROZEN learning rate :",
    FROZEN_MULTIMAE_LR
)

print(
    "FROZEN epochs        :",
    FROZEN_MULTIMAE_EPOCHS
)

print(
    "FROZEN weight decay  :",
    FROZEN_MULTIMAE_WEIGHT_DECAY
)

print(
    "FROZEN batch size    :",
    FROZEN_MULTIMAE_BATCH_SIZE
)


# -----------------------------------------------------------------------------
# 9. Save development selection
# -----------------------------------------------------------------------------

SELECTION_FILE = (
    TUNING_DIR
    / "MultiMAE_Fold3_development_selection.csv"
)


selection_df.to_csv(
    SELECTION_FILE,
    index=False
)


FROZEN_CONFIG_FILE = (
    TUNING_DIR
    / "MultiMAE_Fold3_FROZEN_training_configuration.json"
)


frozen_config = {

    "learning_rate":
        FROZEN_MULTIMAE_LR,

    "epochs":
        FROZEN_MULTIMAE_EPOCHS,

    "weight_decay":
        FROZEN_MULTIMAE_WEIGHT_DECAY,

    "batch_size":
        FROZEN_MULTIMAE_BATCH_SIZE,

    "train_seed":
        TRAIN_SEED,

    "validation_mask_seed":
        VAL_MASK_SEED,

    "development_fold":
        3,

    "selection_metric":
        (
            "minimum Fold-3 masked "
            "reconstruction validation MSE"
        ),

    "outer_fold_rule":
        (
            "Freeze LR and epoch count before "
            "outer-fold evaluation; no outer "
            "validation checkpoint selection."
        )
}


with open(
    FROZEN_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        frozen_config,
        f,
        indent=4
    )


print()
print(
    "Selection table saved:"
)

print(
    SELECTION_FILE
)

print()
print(
    "Frozen configuration saved:"
)

print(
    FROZEN_CONFIG_FILE
)


print()
print("=" * 96)
print(
    "STATUS: MULTIMAE DEVELOPMENT TRAINING SWEEP PASSED"
)
print("=" * 96)

In [ ]:
# =============================================================================
# Paper1_11_MultiMAE_Baseline_13x13
# CELL 4
#
# Load selected Fold-3 MultiMAE-style checkpoint
# Extract deterministic FULL-INPUT 64-D embeddings
#
# No masking is used during representation extraction.
# Raw 64-D latent vectors are retained without latent standardization,
# matching the PB-MMAE and Vanilla MAE clustering protocol.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader


# -----------------------------------------------------------------------------
# 1. Locate frozen selected checkpoint
# -----------------------------------------------------------------------------

selection_file = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "MultiMAE_training_tuning"
    / "MultiMAE_Fold3_development_selection.csv"
)

assert selection_file.exists(), selection_file


selection_df_reload = pd.read_csv(
    selection_file
)


winner = (
    selection_df_reload
    .sort_values(
        "Validation_MSE_at_best_epoch",
        ascending=True
    )
    .iloc[0]
)


SELECTED_LR = float(
    winner[
        "Learning_rate"
    ]
)

SELECTED_EPOCH = int(
    winner[
        "Best_epoch"
    ]
)

SELECTED_CHECKPOINT = Path(
    winner[
        "Checkpoint"
    ]
)


assert SELECTED_LR == 1e-3
assert SELECTED_EPOCH == 57
assert SELECTED_CHECKPOINT.exists(), SELECTED_CHECKPOINT


print("=" * 96)
print("MULTIMAE-STYLE — SELECTED FOLD-3 CHECKPOINT")
print("=" * 96)

print(
    "Learning rate :",
    SELECTED_LR
)

print(
    "Best epoch    :",
    SELECTED_EPOCH
)

print(
    "Checkpoint    :",
    SELECTED_CHECKPOINT
)


# -----------------------------------------------------------------------------
# 2. Rebuild architecture
# -----------------------------------------------------------------------------

model = build_fresh_multimae()


checkpoint = torch.load(
    SELECTED_CHECKPOINT,
    map_location=DEVICE
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


model.eval()


print()
print(
    "Checkpoint loaded successfully."
)


# -----------------------------------------------------------------------------
# 3. Deterministic non-shuffled loaders
# -----------------------------------------------------------------------------

train_embed_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

val_embed_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)


# -----------------------------------------------------------------------------
# 4. Full-input embedding extraction
# -----------------------------------------------------------------------------

@torch.no_grad()
def extract_multimae_embeddings(
    model,
    loader
):

    model.eval()

    latent_parts = []


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )


        z = model.encode_full(
            batch
        )


        latent_parts.append(
            z.detach()
            .cpu()
            .numpy()
        )


    return np.concatenate(
        latent_parts,
        axis=0
    ).astype(
        np.float32
    )


Z_train_mm = (
    extract_multimae_embeddings(
        model,
        train_embed_loader
    )
)

Z_val_mm = (
    extract_multimae_embeddings(
        model,
        val_embed_loader
    )
)


# -----------------------------------------------------------------------------
# 5. Hard shape / numerical checks
# -----------------------------------------------------------------------------

assert Z_train_mm.shape == (
    455,
    64
)

assert Z_val_mm.shape == (
    392,
    64
)

assert np.isfinite(
    Z_train_mm
).all()

assert np.isfinite(
    Z_val_mm
).all()


print()
print("=" * 96)
print("FULL-INPUT EMBEDDING EXTRACTION")
print("=" * 96)

print(
    "TRAIN shape:",
    Z_train_mm.shape
)

print(
    "VAL shape  :",
    Z_val_mm.shape
)

print(
    "TRAIN finite:",
    np.isfinite(
        Z_train_mm
    ).all()
)

print(
    "VAL finite  :",
    np.isfinite(
        Z_val_mm
    ).all()
)


# -----------------------------------------------------------------------------
# 6. Latent-space audit
# -----------------------------------------------------------------------------

def latent_audit(
    Z
):

    dim_std = Z.std(
        axis=0,
        ddof=1
    )

    vector_norm = np.linalg.norm(
        Z,
        axis=1
    )

    return {

        "Global_mean":
            float(
                Z.mean()
            ),

        "Global_std":
            float(
                Z.std(
                    ddof=1
                )
            ),

        "Median_dimension_SD":
            float(
                np.median(
                    dim_std
                )
            ),

        "Mean_vector_norm":
            float(
                np.mean(
                    vector_norm
                )
            ),

        "Median_vector_norm":
            float(
                np.median(
                    vector_norm
                )
            ),

        "Near_zero_SD_dimensions":
            int(
                np.sum(
                    dim_std
                    <
                    1e-6
                )
            ),
    }


train_audit = latent_audit(
    Z_train_mm
)

val_audit = latent_audit(
    Z_val_mm
)


print()
print("=" * 96)
print("TRAIN LATENT AUDIT")
print("=" * 96)

for key, value in (
    train_audit.items()
):

    if isinstance(
        value,
        float
    ):

        print(
            f"{key:<28}: "
            f"{value:.6f}"
        )

    else:

        print(
            f"{key:<28}: "
            f"{value}"
        )


print()
print("=" * 96)
print("VALIDATION LATENT AUDIT")
print("=" * 96)

for key, value in (
    val_audit.items()
):

    if isinstance(
        value,
        float
    ):

        print(
            f"{key:<28}: "
            f"{value:.6f}"
        )

    else:

        print(
            f"{key:<28}: "
            f"{value}"
        )


# -----------------------------------------------------------------------------
# 7. Validation / TRAIN latent-SD transfer
# -----------------------------------------------------------------------------

train_dim_std = Z_train_mm.std(
    axis=0,
    ddof=1
)

val_dim_std = Z_val_mm.std(
    axis=0,
    ddof=1
)


valid_dims = (
    train_dim_std
    >
    1e-6
)


sd_ratio = (
    val_dim_std[
        valid_dims
    ]
    /
    train_dim_std[
        valid_dims
    ]
)


mean_sd_ratio = float(
    np.mean(
        sd_ratio
    )
)

median_sd_ratio = float(
    np.median(
        sd_ratio
    )
)

sd_sd_ratio = float(
    np.std(
        sd_ratio,
        ddof=1
    )
)


print()
print("=" * 96)
print("FOLD-3 LATENT TRANSFER")
print("=" * 96)

print(
    "Mean VAL/TRAIN SD ratio  :",
    f"{mean_sd_ratio:.6f}"
)

print(
    "Median VAL/TRAIN SD ratio:",
    f"{median_sd_ratio:.6f}"
)

print(
    "SD of dimension ratios   :",
    f"{sd_sd_ratio:.6f}"
)

print(
    "Collapsed TRAIN dims     :",
    int(
        np.sum(
            train_dim_std
            <
            1e-6
        )
    )
)

print(
    "Collapsed VAL dims       :",
    int(
        np.sum(
            val_dim_std
            <
            1e-6
        )
    )
)


# -----------------------------------------------------------------------------
# 8. Save embeddings
# -----------------------------------------------------------------------------

EMBED_DIR = (
    OUTPUT_ROOT
    / "Fold_3_DEVELOPMENT"
    / "embeddings"
)

EMBED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_EMBED_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_TRAIN_13x13_latent64.npy"
)

VAL_EMBED_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_VALIDATION_13x13_latent64.npy"
)


np.save(
    TRAIN_EMBED_FILE,
    Z_train_mm
)

np.save(
    VAL_EMBED_FILE,
    Z_val_mm
)


# -----------------------------------------------------------------------------
# 9. Save matching metadata
# -----------------------------------------------------------------------------

TRAIN_META_OUT = (
    EMBED_DIR
    / "MultiMAE_Fold3_TRAIN_13x13_metadata.csv"
)

VAL_META_OUT = (
    EMBED_DIR
    / "MultiMAE_Fold3_VALIDATION_13x13_metadata.csv"
)


train_meta.to_csv(
    TRAIN_META_OUT,
    index=False
)

val_meta.to_csv(
    VAL_META_OUT,
    index=False
)


# -----------------------------------------------------------------------------
# 10. Save latent audit
# -----------------------------------------------------------------------------

latent_audit_df = pd.DataFrame(
    [
        {
            "Split":
                "TRAIN",

            **train_audit
        },

        {
            "Split":
                "VALIDATION",

            **val_audit
        }
    ]
)


LATENT_AUDIT_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_latent_audit.csv"
)


latent_audit_df.to_csv(
    LATENT_AUDIT_FILE,
    index=False
)


transfer_audit_df = pd.DataFrame(
    [
        {
            "Mean_VAL_to_TRAIN_SD_ratio":
                mean_sd_ratio,

            "Median_VAL_to_TRAIN_SD_ratio":
                median_sd_ratio,

            "SD_VAL_to_TRAIN_SD_ratio":
                sd_sd_ratio,

            "TRAIN_near_zero_SD_dims":
                int(
                    np.sum(
                        train_dim_std
                        <
                        1e-6
                    )
                ),

            "VALIDATION_near_zero_SD_dims":
                int(
                    np.sum(
                        val_dim_std
                        <
                        1e-6
                    )
                ),
        }
    ]
)


TRANSFER_AUDIT_FILE = (
    EMBED_DIR
    / "MultiMAE_Fold3_latent_transfer_audit.csv"
)


transfer_audit_df.to_csv(
    TRANSFER_AUDIT_FILE,
    index=False
)


# -----------------------------------------------------------------------------
# 11. Final hard checks
# -----------------------------------------------------------------------------

assert (
    train_audit[
        "Near_zero_SD_dimensions"
    ]
    ==
    0
)

assert (
    val_audit[
        "Near_zero_SD_dimensions"
    ]
    ==
    0
)


print()
print("=" * 96)
print(
    "STATUS: MULTIMAE FOLD-3 EMBEDDING EXTRACTION PASSED"
)
print("=" * 96)

print()
print(
    "TRAIN embeddings:"
)

print(
    TRAIN_EMBED_FILE
)

print()
print(
    "VALIDATION embeddings:"
)

print(
    VAL_EMBED_FILE
)

print()
print(
    "Latent audit:"
)

print(
    LATENT_AUDIT_FILE
)

In [ ]:
# =============================================================================
# CELL 5
# MultiMAE-style — FROZEN OUTER-FOLD TRAINING
#
# Outer folds: 1, 2, 4, 5
#
# FROZEN FROM FOLD-3 DEVELOPMENT:
#   LR           = 1e-3
#   Epochs       = 57
#   Weight decay = 1e-4
#   Batch size   = 32
#
# IMPORTANT:
#   - No outer-fold tuning.
#   - No early stopping.
#   - No checkpoint selection from outer validation loss.
#   - Final epoch 57 is retained for every outer fold.
# =============================================================================

from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)


# -----------------------------------------------------------------------------
# 1. Frozen configuration
# -----------------------------------------------------------------------------

OUTER_FOLDS = [
    1,
    2,
    4,
    5
]

FROZEN_MM_LR = 1e-3
FROZEN_MM_EPOCHS = 57
FROZEN_MM_WEIGHT_DECAY = 1e-4
FROZEN_MM_BATCH_SIZE = 32

OUTER_BASE_SEED = 2026

VAL_MASK_SEED = 123456


print("=" * 100)
print("MULTIMAE-STYLE — FROZEN OUTER-FOLD TRAINING")
print("=" * 100)

print("Outer folds       :", OUTER_FOLDS)
print("Learning rate     :", FROZEN_MM_LR)
print("Epochs            :", FROZEN_MM_EPOCHS)
print("Weight decay      :", FROZEN_MM_WEIGHT_DECAY)
print("Batch size        :", FROZEN_MM_BATCH_SIZE)
print(
    "Masked/modality   :",
    MM_N_MASKED_PER_MODALITY
)
print("Latent dimension  :", MM_EMBED_DIM)
print("Device            :", DEVICE)


# -----------------------------------------------------------------------------
# 2. Dataset helper
# -----------------------------------------------------------------------------

class FrozenMultiMAEDataset(
    Dataset
):

    def __init__(
        self,
        npy_path
    ):

        self.x = np.load(
            npy_path,
            mmap_mode="r"
        )


    def __len__(
        self
    ):

        return self.x.shape[0]


    def __getitem__(
        self,
        idx
    ):

        sample = np.array(
            self.x[idx],
            dtype=np.float32,
            copy=True
        )

        return torch.from_numpy(
            sample
        )


# -----------------------------------------------------------------------------
# 3. Reproducibility
# -----------------------------------------------------------------------------

def reset_outer_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# 4. Fresh model constructor
# -----------------------------------------------------------------------------

def build_outer_multimae():

    model = MultiMAEStyle13x13(

        patch_size=PATCH_SIZE,

        modality_channels=(
            MODALITY_CHANNELS
        ),

        embed_dim=MM_EMBED_DIM,

        encoder_depth=(
            MM_ENCODER_DEPTH
        ),

        encoder_heads=(
            MM_ENCODER_HEADS
        ),

        encoder_mlp_dim=(
            MM_ENCODER_MLP_DIM
        ),

        decoder_dim=(
            MM_DECODER_DIM
        ),

        decoder_depth=(
            MM_DECODER_DEPTH
        ),

        decoder_heads=(
            MM_DECODER_HEADS
        ),

        decoder_mlp_dim=(
            MM_DECODER_MLP_DIM
        ),

        n_masked_per_modality=(
            MM_N_MASKED_PER_MODALITY
        ),

        dropout=MM_DROPOUT
    )

    return model.to(
        DEVICE
    )


# -----------------------------------------------------------------------------
# 5. Diagnostic fixed-mask validation loss
#
# Used for reporting only.
# NEVER used to choose an outer-fold checkpoint.
# -----------------------------------------------------------------------------

@torch.no_grad()
def outer_validation_loss(
    model,
    loader,
    seed=VAL_MASK_SEED
):

    model.eval()


    cpu_state = (
        torch.get_rng_state()
    )


    cuda_state = None

    if torch.cuda.is_available():

        cuda_state = (
            torch.cuda.get_rng_state_all()
        )


    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    total_loss = 0.0
    total_n = 0


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )


        output = model(
            batch
        )


        n = batch.shape[0]


        total_loss += (
            float(
                output["loss"].item()
            )
            *
            n
        )


        total_n += n


    torch.set_rng_state(
        cpu_state
    )


    if (
        torch.cuda.is_available()
        and cuda_state is not None
    ):

        torch.cuda.set_rng_state_all(
            cuda_state
        )


    return (
        total_loss
        /
        total_n
    )


# -----------------------------------------------------------------------------
# 6. Full-input 64-D representation extraction
# -----------------------------------------------------------------------------

@torch.no_grad()
def extract_outer_embeddings(
    model,
    loader
):

    model.eval()

    latent_parts = []


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )


        z = model.encode_full(
            batch
        )


        latent_parts.append(
            z.detach()
            .cpu()
            .numpy()
        )


    return np.concatenate(
        latent_parts,
        axis=0
    ).astype(
        np.float32
    )


# -----------------------------------------------------------------------------
# 7. Outer-fold loop
# -----------------------------------------------------------------------------

outer_records = []


for fold in OUTER_FOLDS:

    print()
    print("=" * 100)
    print(f"OUTER FOLD {fold}")
    print("=" * 100)


    fold_seed = (
        OUTER_BASE_SEED
        +
        fold
    )


    reset_outer_seed(
        fold_seed
    )


    # -------------------------------------------------------------------------
    # Paths
    # -------------------------------------------------------------------------

    fold_dir = (
        PATCH_ROOT
        / f"Fold_{fold}"
    )


    train_file = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"TRAIN_13x13_14ch.npy"
        )
    )


    val_file = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"VALIDATION_13x13_14ch.npy"
        )
    )


    train_meta_file = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"TRAIN_13x13_metadata.csv"
        )
    )


    val_meta_file = (
        fold_dir
        / (
            f"Fold{fold}_"
            f"VALIDATION_13x13_metadata.csv"
        )
    )


    for path in [
        train_file,
        val_file,
        train_meta_file,
        val_meta_file
    ]:

        assert path.exists(), path


    # -------------------------------------------------------------------------
    # Dataset + loaders
    # -------------------------------------------------------------------------

    train_ds_outer = (
        FrozenMultiMAEDataset(
            train_file
        )
    )


    val_ds_outer = (
        FrozenMultiMAEDataset(
            val_file
        )
    )


    generator = (
        torch.Generator()
    )

    generator.manual_seed(
        fold_seed
    )


    train_loader = DataLoader(

        train_ds_outer,

        batch_size=(
            FROZEN_MM_BATCH_SIZE
        ),

        shuffle=True,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        generator=generator,

        drop_last=False
    )


    val_loader = DataLoader(

        val_ds_outer,

        batch_size=(
            FROZEN_MM_BATCH_SIZE
        ),

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False
    )


    # -------------------------------------------------------------------------
    # Fresh independently initialized model
    # -------------------------------------------------------------------------

    reset_outer_seed(
        fold_seed
    )


    model = (
        build_outer_multimae()
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=FROZEN_MM_LR,

        weight_decay=(
            FROZEN_MM_WEIGHT_DECAY
        )
    )


    # -------------------------------------------------------------------------
    # Train EXACTLY 57 epochs
    # -------------------------------------------------------------------------

    history = []

    start_time = (
        time.time()
    )


    for epoch in range(
        1,
        FROZEN_MM_EPOCHS + 1
    ):

        model.train()


        total_train_loss = 0.0
        total_train_n = 0


        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            output = model(
                batch
            )


            loss = output[
                "loss"
            ]


            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    f"Non-finite loss: "
                    f"Fold={fold}, "
                    f"Epoch={epoch}"
                )


            loss.backward()

            optimizer.step()


            n = batch.shape[0]


            total_train_loss += (
                float(
                    loss.item()
                )
                *
                n
            )


            total_train_n += n


        train_loss = (
            total_train_loss
            /
            total_train_n
        )


        # Diagnostic only
        val_loss = (
            outer_validation_loss(
                model,
                val_loader,
                seed=VAL_MASK_SEED
            )
        )


        history.append(
            {
                "Fold":
                    fold,

                "Epoch":
                    epoch,

                "Train_masked_MSE":
                    train_loss,

                "Validation_masked_MSE":
                    val_loss
            }
        )


        if (
            epoch == 1
            or
            epoch % 10 == 0
            or
            epoch == FROZEN_MM_EPOCHS
        ):

            print(
                f"Epoch "
                f"{epoch:02d}/"
                f"{FROZEN_MM_EPOCHS} | "
                f"train="
                f"{train_loss:.8f} | "
                f"val="
                f"{val_loss:.8f}"
            )


    elapsed = (
        time.time()
        -
        start_time
    )


    # -------------------------------------------------------------------------
    # Output directories
    # -------------------------------------------------------------------------

    fold_output = (
        OUTPUT_ROOT
        / f"Fold_{fold}_OUTER"
    )


    embedding_dir = (
        fold_output
        / "embeddings"
    )


    fold_output.mkdir(
        parents=True,
        exist_ok=True
    )


    embedding_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # -------------------------------------------------------------------------
    # Save FINAL epoch-57 model
    # -------------------------------------------------------------------------

    checkpoint_file = (
        fold_output
        / (
            f"MultiMAE_Fold{fold}_"
            f"13x13_final_epoch"
            f"{FROZEN_MM_EPOCHS}.pt"
        )
    )


    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "fold":
                fold,

            "learning_rate":
                FROZEN_MM_LR,

            "epochs":
                FROZEN_MM_EPOCHS,

            "weight_decay":
                FROZEN_MM_WEIGHT_DECAY,

            "batch_size":
                FROZEN_MM_BATCH_SIZE,

            "seed":
                fold_seed,

            "selection_rule":
                (
                    "Fold-3-frozen training "
                    "configuration; final epoch "
                    "retained with no outer "
                    "validation checkpoint "
                    "selection."
                )
        },
        checkpoint_file
    )


    # -------------------------------------------------------------------------
    # Save training history
    # -------------------------------------------------------------------------

    history_df = pd.DataFrame(
        history
    )


    history_file = (
        fold_output
        / (
            f"MultiMAE_Fold{fold}_"
            f"training_history.csv"
        )
    )


    history_df.to_csv(
        history_file,
        index=False
    )


    # -------------------------------------------------------------------------
    # Non-shuffled loaders for embedding extraction
    # -------------------------------------------------------------------------

    train_embedding_loader = DataLoader(

        train_ds_outer,

        batch_size=64,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False
    )


    val_embedding_loader = DataLoader(

        val_ds_outer,

        batch_size=64,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            torch.cuda.is_available()
        ),

        drop_last=False
    )


    # -------------------------------------------------------------------------
    # Full-input embeddings
    # -------------------------------------------------------------------------

    Z_train_outer = (
        extract_outer_embeddings(
            model,
            train_embedding_loader
        )
    )


    Z_val_outer = (
        extract_outer_embeddings(
            model,
            val_embedding_loader
        )
    )


    assert Z_train_outer.shape == (
        len(train_ds_outer),
        64
    )


    assert Z_val_outer.shape == (
        len(val_ds_outer),
        64
    )


    assert np.isfinite(
        Z_train_outer
    ).all()


    assert np.isfinite(
        Z_val_outer
    ).all()


    # -------------------------------------------------------------------------
    # Save embeddings
    # -------------------------------------------------------------------------

    train_embedding_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"TRAIN_13x13_latent64.npy"
        )
    )


    val_embedding_file = (
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"VALIDATION_13x13_latent64.npy"
        )
    )


    np.save(
        train_embedding_file,
        Z_train_outer
    )


    np.save(
        val_embedding_file,
        Z_val_outer
    )


    # -------------------------------------------------------------------------
    # Preserve matching metadata order
    # -------------------------------------------------------------------------

    train_meta_outer = (
        pd.read_csv(
            train_meta_file
        )
    )


    val_meta_outer = (
        pd.read_csv(
            val_meta_file
        )
    )


    assert len(
        train_meta_outer
    ) == len(
        Z_train_outer
    )


    assert len(
        val_meta_outer
    ) == len(
        Z_val_outer
    )


    train_meta_outer.to_csv(
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"TRAIN_13x13_metadata.csv"
        ),
        index=False
    )


    val_meta_outer.to_csv(
        embedding_dir
        / (
            f"MultiMAE_Fold{fold}_"
            f"VALIDATION_13x13_metadata.csv"
        ),
        index=False
    )


    # -------------------------------------------------------------------------
    # Latent transfer audit
    # -------------------------------------------------------------------------

    train_std = (
        Z_train_outer.std(
            axis=0,
            ddof=1
        )
    )


    val_std = (
        Z_val_outer.std(
            axis=0,
            ddof=1
        )
    )


    valid_dims = (
        train_std
        >
        1e-6
    )


    sd_ratio = (
        val_std[
            valid_dims
        ]
        /
        train_std[
            valid_dims
        ]
    )


    record = {

        "Fold":
            fold,

        "Train_n":
            len(
                train_ds_outer
            ),

        "Validation_n":
            len(
                val_ds_outer
            ),

        "Final_train_MSE":
            float(
                history_df.iloc[-1][
                    "Train_masked_MSE"
                ]
            ),

        "Final_validation_MSE":
            float(
                history_df.iloc[-1][
                    "Validation_masked_MSE"
                ]
            ),

        "Mean_VAL_to_TRAIN_SD_ratio":
            float(
                np.mean(
                    sd_ratio
                )
            ),

        "Median_VAL_to_TRAIN_SD_ratio":
            float(
                np.median(
                    sd_ratio
                )
            ),

        "TRAIN_near_zero_SD_dims":
            int(
                np.sum(
                    train_std
                    <
                    1e-6
                )
            ),

        "VALIDATION_near_zero_SD_dims":
            int(
                np.sum(
                    val_std
                    <
                    1e-6
                )
            ),

        "Training_seconds":
            float(
                elapsed
            )
    }


    outer_records.append(
        record
    )


    print()
    print(
        "TRAIN embedding      :",
        Z_train_outer.shape
    )

    print(
        "VALIDATION embedding :",
        Z_val_outer.shape
    )

    print(
        "Mean VAL/TRAIN SD    :",
        f"{record['Mean_VAL_to_TRAIN_SD_ratio']:.4f}"
    )

    print(
        "Training time        :",
        f"{elapsed:.2f} s"
    )


# -----------------------------------------------------------------------------
# 8. Outer-fold summary
# -----------------------------------------------------------------------------

outer_summary_df = pd.DataFrame(
    outer_records
)


OUTER_SUMMARY_FILE = (
    OUTPUT_ROOT
    / "MultiMAE_13x13_outer_fold_training_summary.csv"
)


outer_summary_df.to_csv(
    OUTER_SUMMARY_FILE,
    index=False
)


print()
print("=" * 100)
print("MULTIMAE-STYLE — OUTER-FOLD TRAINING SUMMARY")
print("=" * 100)


display(
    outer_summary_df.round(
        4
    )
)


print()
print(
    "Mean outer VAL/TRAIN SD ratio:",
    round(
        outer_summary_df[
            "Mean_VAL_to_TRAIN_SD_ratio"
        ].mean(),
        4
    )
)


# -----------------------------------------------------------------------------
# 9. Hard checks
# -----------------------------------------------------------------------------

assert set(
    outer_summary_df[
        "Fold"
    ]
) == {
    1,
    2,
    4,
    5
}


assert (
    outer_summary_df[
        "TRAIN_near_zero_SD_dims"
    ]
    ==
    0
).all()


assert (
    outer_summary_df[
        "VALIDATION_near_zero_SD_dims"
    ]
    ==
    0
).all()


print()
print("=" * 100)
print(
    "STATUS: FROZEN MULTIMAE OUTER-FOLD TRAINING PASSED"
)
print("=" * 100)

print(
    "All outer models used the Fold-3-frozen "
    "LR=1e-3 and 57-epoch training budget. "
    "No outer validation checkpoint selection was performed."
)